# DSE Data Cleaning

Works on **Google Colab** (file upload) and **locally** (CSV path). Cleaning is done **per stock** so high-priced tickers are not clipped by the global 99.9th percentile.

In [ ]:
# Optional on Colab
# %pip install -q pandas numpy scipy plotly

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import skew
from IPython.display import display

try:
    import plotly.express as px
    HAS_PLOTLY = True
except ImportError:
    HAS_PLOTLY = False
    import matplotlib.pyplot as plt


def load_raw_dse():
    candidates = [
        Path("data/DSE_raw.csv"),
        Path("../data/DSE_raw.csv"),
        Path("Cleaned.csv"),
        Path("Cleaned_DSE_Data.csv"),
        Path("../Cleaned.csv"),
    ]
    for path in candidates:
        if path.exists():
            print(f"Loaded {path.resolve()}")
            return pd.read_csv(path, low_memory=False)

    try:
        from google.colab import files
        uploaded = files.upload()
        name = next(iter(uploaded))
        return pd.read_csv(name, low_memory=False)
    except Exception as exc:
        raise FileNotFoundError(
            "Place the Mendeley DSE CSV as data/DSE_raw.csv or upload it in Colab."
        ) from exc


df_raw = load_raw_dse()
df_raw.head()

In [ ]:
df = df_raw.copy()
df.columns = [c.strip().replace(" ", "_") for c in df.columns]

rename = {}
for col in df.columns:
    low = col.lower()
    if low in {"tradingcode", "trading_code", "ticker", "symbol"}:
        rename[col] = "Trading_Code"
    elif low == "date":
        rename[col] = "Date"
    elif low in {"open", "opening"}:
        rename[col] = "Open"
    elif low in {"high"}:
        rename[col] = "High"
    elif low in {"low"}:
        rename[col] = "Low"
    elif low in {"close", "closing", "ltp"}:
        rename[col] = "Close"
    elif "volume" in low:
        rename[col] = "Volume"
df = df.rename(columns=rename)

needed = ["Trading_Code", "Date", "Open", "High", "Low", "Close", "Volume"]
missing = [c for c in needed if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns {missing}. Found: {list(df.columns)}")

df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df = df.dropna(subset=["Date", "Trading_Code"])
for col in ["Open", "High", "Low", "Close", "Volume"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=["Open", "High", "Low", "Close", "Volume"])
df = df[~((df["Open"] == 0) & (df["High"] == 0) & (df["Low"] == 0) & (df["Close"] == 0))]
df = df[(df["Close"] > 0) & (df["High"] >= df["Low"])]
df = df.drop_duplicates(subset=["Trading_Code", "Date"])
df = df.sort_values(["Trading_Code", "Date"]).reset_index(drop=True)

print(df.info())
print("Missing:\n", df.isna().sum())

In [ ]:
def clip_outliers_per_stock(group: pd.DataFrame) -> pd.DataFrame:
    g = group.copy()
    for col in ["Open", "High", "Low", "Close", "Volume"]:
        q_low, q_high = g[col].quantile(0.001), g[col].quantile(0.999)
        g[col] = g[col].clip(lower=q_low, upper=q_high)
    return g


df = df.groupby("Trading_Code", group_keys=False).apply(clip_outliers_per_stock)

stock_lengths = df.groupby("Trading_Code").size()
valid = stock_lengths[stock_lengths >= 250].index
df = df[df["Trading_Code"].isin(valid)].reset_index(drop=True)

print("Usable stocks:", df["Trading_Code"].nunique())
print("Rows:", len(df))
print("Date range:", df["Date"].min(), "→", df["Date"].max())

In [ ]:
out_dir = Path("data")
out_dir.mkdir(exist_ok=True)
out_path = out_dir / "Cleaned_DSE_Data.csv"
df.to_csv(out_path, index=False)
df.to_csv("Cleaned.csv", index=False)
print("Saved", out_path.resolve(), "and Cleaned.csv")

try:
    from google.colab import files
    files.download(str(out_path))
except ImportError:
    pass

In [ ]:
summary = pd.DataFrame({
    "Variable": ["Closing Price", "Volume", "Log Close", "Log Volume"],
    "Max": [df["Close"].max(), df["Volume"].max(), np.log1p(df["Close"]).max(), np.log1p(df["Volume"]).max()],
    "Min": [df["Close"].min(), df["Volume"].min(), np.log1p(df["Close"]).min(), np.log1p(df["Volume"]).min()],
    "Mean": [df["Close"].mean(), df["Volume"].mean(), np.log1p(df["Close"]).mean(), np.log1p(df["Volume"]).mean()],
    "SD": [df["Close"].std(), df["Volume"].std(), np.log1p(df["Close"]).std(), np.log1p(df["Volume"]).std()],
    "Skewness": [skew(df["Close"]), skew(df["Volume"]), skew(np.log1p(df["Close"])), skew(np.log1p(df["Volume"]))],
})
display(summary.round(3))

In [ ]:
monthly = (
    df.assign(Month=df["Date"].dt.to_period("M").astype(str))
      .groupby("Month", as_index=False)
      .agg(Close=("Close", "mean"), Volume=("Volume", "sum"))
)

if HAS_PLOTLY:
    px.line(monthly, x="Month", y="Close", title="Average monthly closing price").show()
    px.line(monthly, x="Month", y="Volume", title="Total monthly volume", log_y=True).show()
    px.imshow(df[["Open", "High", "Low", "Close", "Volume"]].corr(), text_auto=True,
              color_continuous_scale="Blues", title="Price/volume correlation").show()
else:
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(pd.to_datetime(monthly["Month"]), monthly["Close"])
    ax.set_title("Average monthly closing price")
    plt.show()